### Tool Usage & The ReAct Loop (Reason + Act) - The Heart of Agency 

Single tool call to multi-step reasoning.

In [ ]:
# run This cell if you are in Google Colab

import os, datetime, shutil
from google.colab import drive

# Mount Drive
# drive.mount('/content/drive', force_remount=True)


In [2]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
api_key = os.getenv("gsk_WORKSHOP_KEY")

client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # This prints the FULL error details
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")
        
        # If it's an API error, get more details
        if hasattr(e, 'response'):
            print(f"Status Code: {e.response.status_code}")
            print(f"Response Body: {e.response.text}")
        return None

### Why Tool usage or Function support is needed ?

In [3]:
# Test
result = ask_llm("Whats the Weather in Bangalore ?'")
if result:
    print(f"SUCCESS: {result}")

SUCCESS: However, I'm a large language model, I don't have real-time access to current weather conditions. But I can suggest some ways to find out the current weather in Bangalore.

You can try checking the following options:

1. **Online Weather Websites**: You can check websites like accuweather.com, weather.com, or wunderground.com for the current weather conditions in Bangalore.
2. **Met Office Website**: The India Meteorological Department (IMD) website provides weather forecasts and weather warnings for various cities in India, including Bangalore.
3. **Social Media**: You can also check the social media handles of your local weather department or news channels for updates on the weather in Bangalore.
4. **Mobile Apps**: You can download mobile apps like Dark Sky, Weather Underground, or AccuWeather to get real-time weather updates for Bangalore.

Please note that the weather can change quickly, so it's always a good idea to check multiple sources for the most up-to-date informat

### Giving the Agent Hands : tool Usage 
Introducing the Function Definition (JSON Schema).
"This is the instruction manual we give the LLM so 

In [4]:
# Tool Definitions
import json

# 1. Define the Python Tool
def get_weather(city: str):
    # Mock database for workshop
    mock_db = {"mumbai": "32C, Humid", "delhi": "28C, Smoggy", "bangalore": "27C, Sultry"}
    return mock_db.get(city.lower(), "Data not available")

# 2. Write the TOOL SCHEMA (This is the tricky part students learn)
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# 3. The AGENTIC CALL
def agent_ask(user_query):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": user_query}],
        tools=tool_schema, # <--- THE MAGIC LINE
        tool_choice="auto"
    )
    return response


### The ReAct Loop (Reason + Act) - The Heart of Agency 

Goal: Move from single tool call to multi-step reasoning.

In [5]:

def run_agent(user_query):
    messages = [{"role": "user", "content": user_query}]
    
    # The Loop - Students fill in the condition
    while True:  # <--- STUDENTS FILL THIS
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tool_schema,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        messages.append(response_message) # Append assistant thought
        
        # Check if LLM wants to call a tool
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                # EXECUTE THE PYTHON FUNCTION (Student fills this)
                if function_name == "get_weather":
                    result = get_weather(function_args.get("city"))
                    # Feed result back to LLM
                    messages.append({
                        "role": "tool",
                        "content": result,
                        "tool_call_id": tool_call.id
                    })
        else:
            # NO MORE TOOLS -> ANSWER IS READY
            return response_message.content



In [6]:
print(run_agent("Is it warmer in Mumbai or Delhi right now?"))

To answer your question, we need to determine which one is warmer between 32C and 28C. In this case, it is warmer in Mumbai right now.
